In [28]:
import cv2
import numpy as np
import math
import pandas as pd

# Load the video file
video_path = 'output_sequences/sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Initialize variables for different parameter sets
average_vx_list_lk = []
average_vy_list_lk = []
V_list_lk = []

average_vx_list_corners_1 = []
average_vy_list_corners_1 = []
V_list_corners_1 = []

average_vx_list_corners_2 = []
average_vy_list_corners_2 = []
V_list_corners_2 = []

average_vx_list_corners_3 = []
average_vy_list_corners_3 = []
V_list_corners_3 = []

average_vx_list_quality_1 = []
average_vy_list_quality_1 = []
V_list_quality_1 = []

average_vx_list_quality_2 = []
average_vy_list_quality_2 = []
V_list_quality_2 = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Different parameters for Lucas-Kanade optical flow
    lk_params = dict(winSize=(25, 25), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 15, 0.01))

    # Different parameters for goodFeaturesToTrack
    p0_lk = cv2.goodFeaturesToTrack(gray1, maxCorners=100, qualityLevel=0.01, minDistance=10)

    p0_corners_1 = cv2.goodFeaturesToTrack(gray1, maxCorners=500, qualityLevel=0.01, minDistance=7)
    p0_corners_2 = cv2.goodFeaturesToTrack(gray1, maxCorners=200, qualityLevel=0.02, minDistance=5)
    p0_corners_3 = cv2.goodFeaturesToTrack(gray1, maxCorners=300, qualityLevel=0.03, minDistance=10)

    p0_quality_1 = cv2.goodFeaturesToTrack(gray1, maxCorners=100, qualityLevel=0.9, minDistance=7)
    p0_quality_2 = cv2.goodFeaturesToTrack(gray1, maxCorners=150, qualityLevel=0.8, minDistance=10)

    # Create a mask for drawing purposes
    mask = np.zeros_like(frame1)

    # Optical flow calculation for different parameter sets
    for p0, vx_list, vy_list, V_list in zip([p0_lk, p0_corners_1, p0_corners_2, p0_corners_3, p0_quality_1, p0_quality_2],
                                            [average_vx_list_lk, average_vx_list_corners_1, average_vx_list_corners_2,
                                             average_vx_list_corners_3, average_vx_list_quality_1, average_vx_list_quality_2],
                                            [average_vy_list_lk, average_vy_list_corners_1, average_vy_list_corners_2,
                                             average_vy_list_corners_3, average_vy_list_quality_1, average_vy_list_quality_2],
                                            [V_list_lk, V_list_corners_1, V_list_corners_2, V_list_corners_3,
                                             V_list_quality_1, V_list_quality_2]):

        # Calculate optical flow from the previous frame to the current frame
        p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

        # Select good points
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        dt = 1 / 30  # Assuming 30 frames per second

        # Calculate speeds in the x and y axes for each point
        vx_list_frame = []
        vy_list_frame = []

        # Draw the tracks and calculate distances
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = np.int32(new.ravel())
            c, d = np.int32(old.ravel())

            # calculate the speed in x and y axis
            vx = abs((a - c) / dt)
            vy = abs((b - d) / dt)
            vx_list_frame.append(vx)
            vy_list_frame.append(vy)

            mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
            frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)

        # calculate the average speed in x and y axis for the current parameter set
        average_vx = sum(vx_list_frame) / len(vx_list_frame)
        average_vy = sum(vy_list_frame) / len(vy_list_frame)
        V = math.sqrt(pow(average_vx, 2) + pow(average_vy, 2))

        vx_list.append(average_vx)
        vy_list.append(average_vy)
        V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()

# Create DataFrames for different parameter sets
df_lk = pd.DataFrame({'vx_lk': average_vx_list_lk, 'vy_lk': average_vy_list_lk, 'v_lk': V_list_lk})
df_corners_1 = pd.DataFrame({'vx_corners_1': average_vx_list_corners_1, 'vy_corners_1': average_vy_list_corners_1, 'v_corners_1': V_list_corners_1})
df_corners_2 = pd.DataFrame({'vx_corners_2': average_vx_list_corners_2, 'vy_corners_2': average_vy_list_corners_2, 'v_corners_2': V_list_corners_2})
df_corners_3 = pd.DataFrame({'vx_corners_3': average_vx_list_corners_3, 'vy_corners_3': average_vy_list_corners_3, 'v_corners_3': V_list_corners_3})
df_quality_1 = pd.DataFrame({'vx_quality_1': average_vx_list_quality_1, 'vy_quality_1': average_vy_list_quality_1, 'v_quality_1': V_list_quality_1})
df_quality_2 = pd.DataFrame({'vx_quality_2': average_vx_list_quality_2, 'vy_quality_2': average_vy_list_quality_2, 'v_quality_2': V_list_quality_2})




In [29]:
# Display the DataFrames
print("DataFrame for Lucas-Kanade parameters:")
df_lk

DataFrame for Lucas-Kanade parameters:


,vx_lk,vy_lk,v_lk
0,203.333333,130.303030,241.502224
1,212.100000,107.400000,237.741814
2,95.151515,67.272727,116.530814
3,49.200000,48.900000,69.367500
4,41.100000,48.600000,63.648802
...,...,...,...
177,11.700000,119.400000,119.971872
178,36.300000,85.200000,92.610637
179,37.200000,160.500000,164.754636
180,26.100000,81.900000,85.958246


In [30]:
print("\nDataFrame for goodFeaturesToTrack with maxCorners=500, qualityLevel=0.01:")
df_corners_1


DataFrame for goodFeaturesToTrack with maxCorners=500, qualityLevel=0.01:


,vx_corners_1,vy_corners_1,v_corners_1
0,207.813765,128.623482,244.398366
1,206.100000,125.880000,241.501520
2,94.929860,77.314629,122.430512
3,51.600000,53.820000,74.559724
4,48.660000,65.160000,81.324174
...,...,...,...
177,15.120000,110.400000,111.430581
178,36.180000,93.600000,100.349152
179,32.404810,165.991984,169.125428
180,26.700000,85.500000,89.571982


In [31]:
print("\nDataFrame for goodFeaturesToTrack with maxCorners=200, qualityLevel=0.02:")
df_corners_2


DataFrame for goodFeaturesToTrack with maxCorners=200, qualityLevel=0.02:


,vx_corners_2,vy_corners_2,v_corners_2
0,206.984925,128.894472,243.837126
1,211.650000,112.650000,239.761851
2,94.371859,73.718593,119.751738
3,49.350000,47.850000,68.738963
4,42.900000,55.650000,70.266155
...,...,...,...
177,12.600000,118.950000,119.615478
178,37.950000,86.700000,94.641917
179,31.800000,159.900000,163.031439
180,22.650000,77.850000,81.078018


In [32]:
print("\nDataFrame for goodFeaturesToTrack with maxCorners=300, qualityLevel=0.03:")
df_corners_3


DataFrame for goodFeaturesToTrack with maxCorners=300, qualityLevel=0.03:


,vx_corners_3,vy_corners_3,v_corners_3
0,208.489933,128.255034,244.780321
1,207.500000,123.900000,241.676354
2,97.926421,76.153846,124.052377
3,49.900000,51.500000,71.709553
4,45.900000,65.200000,79.736127
...,...,...,...
177,14.100000,112.400000,113.280934
178,36.300000,91.600000,98.530452
179,39.100000,172.000000,176.388237
180,25.300000,84.300000,88.014658


In [33]:
print("\nDataFrame for goodFeaturesToTrack with maxCorners=100, qualityLevel=0.9:")
df_quality_1


DataFrame for goodFeaturesToTrack with maxCorners=100, qualityLevel=0.9:


,vx_quality_1,vy_quality_1,v_quality_1
0,210.0,150.0,258.069758
1,300.0,60.0,305.941171
2,270.0,210.0,342.052628
3,60.0,90.0,108.166538
4,30.0,210.0,212.132034
...,...,...,...
177,0.0,120.0,120.000000
178,30.0,60.0,67.082039
179,210.0,240.0,318.904374
180,0.0,30.0,30.000000


In [34]:
print("\nDataFrame for goodFeaturesToTrack with maxCorners=150, qualityLevel=0.8:")
df_quality_2


DataFrame for goodFeaturesToTrack with maxCorners=150, qualityLevel=0.8:


,vx_quality_2,vy_quality_2,v_quality_2
0,210.0,150.0,258.069758
1,300.0,60.0,305.941171
2,270.0,210.0,342.052628
3,60.0,90.0,108.166538
4,30.0,210.0,212.132034
...,...,...,...
177,15.0,142.5,143.287299
178,30.0,60.0,67.082039
179,210.0,240.0,318.904374
180,0.0,40.0,40.000000


In [35]:
# Save DataFrames to CSV files
df_lk.to_csv('velocity generated/df_lk.csv', index=False)
df_corners_1.to_csv('velocity generated/df_corners_1.csv', index=False)
df_corners_2.to_csv('velocity generated/df_corners_2.csv', index=False)
df_corners_3.to_csv('velocity generated/df_corners_3.csv', index=False)
df_quality_1.to_csv('velocity generated/df_quality_1.csv', index=False)
df_quality_2.to_csv('velocity generated/df_quality_2.csv', index=False)